In [ ]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

# Standard library imports
import os
import sys
import time
import pickle
import random
import copy
from collections import defaultdict

# Third-party imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
from rdkit import Chem
from rdkit.Chem import AllChem, QED, rdMolDescriptors, MolSurf, rdDepictor
from rdkit.Chem.Draw import SimilarityMaps, rdMolDraw2D
from numpy.polynomial.polynomial import polyfit
from tqdm import tqdm
import scipy
import itertools

# Configure PyTorch
torch.manual_seed(8)
sys.setrecursionlimit(50000)
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.nn.Module.dump_patches = True

# Configure matplotlib
%matplotlib inline

# Custom module imports
from GCN import Fingerprint, GCNModel, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 50.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 55.1 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [ ]:
def create_structured_absolute_ecfp_dictionary(df, split_calixarene_dict, holdout_size=0.5):
    """
    Creates structured train/test split for absolute training/prediction.
    
    Args:
        df (pd.DataFrame): Input dataframe containing host and target data
        split_calixarene_dict (dict): Dictionary with 'predictable' and 'unpredictable' host lists
        holdout_size (float): Fraction of data to use for testing (default: 0.5)
    
    Returns:
        tuple: (train_df, test_df) - DataFrames for training and testing
    """
    calixarene_df = df
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 
                      'H3K9me3', 'H3R2me2a', 'H3R2me2s', 'cano_smiles']

    # Determine holdout amounts
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    # Sample holdout calixarenes
    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred
    
    train_data = []
    test_data = []
    
    grouped_df = calixarene_df.groupby('Host')
    
    for host, group in grouped_df:
        row = group.iloc[0]
        
        base_record = {
            'Host': host,
            'SMILES': row['SMILES']
        }
        
        for column in target_columns:
            base_record[column] = row[column]
        
        if host not in all_holdout_calix:
            train_data.append(base_record)
        else:
            test_data.append(base_record)
    
    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)
    
    return train_df, test_df

# Calixarene host classification
split_calix_dict = {
    'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6', 'AP7', 'AP8', 'AP9', 
                    'AH1', 'AH2', 'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3', 'E1', 'E3', 'E6', 
                    'E7', 'E8', 'E11', 'P-NO2', 'PSC4'],
    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1', 'CP2', 'DP2', 
                      'DM1', 'DO2', 'DO3']
}




In [ ]:
def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, batch_size=50, 
                          epochs=800, p_dropout=0.1, fingerprint_dim=150, weight_decay=2, 
                          learning_rate=3, radius=3, T=2, per_target_output_units_num=1):
    """
    Prepares model architecture and processes molecular data for training.
    
    Args:
        raw_filename (str): Path to CSV file containing SMILES and target data
        target_name (str): Name identifier for the target (default: 'Calx')
        targets (list): List of target column names (default: histone modifications)
        batch_size (int): Training batch size (default: 50)
        epochs (int): Number of training epochs (default: 800)
        p_dropout (float): Dropout probability (default: 0.1)
        fingerprint_dim (int): Fingerprint dimension (default: 150)
        weight_decay (int): Weight decay exponent (default: 2)
        learning_rate (int): Learning rate exponent (default: 3)
        radius (int): Molecular fingerprint radius (default: 3)
        T (int): Number of fingerprint iterations (default: 2)
        per_target_output_units_num (int): Output units per target (default: 1)
    
    Returns:
        tuple: (model, optimizer, loss_function, remained_df, targets, feature_dicts)
    """
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 
                   'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print(f"Number of SMILES molecules: {len(smilesList)}")

    remained_smiles = []
    canonical_smiles_list = []
    
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                remained_smiles.append(smiles)
                canonical_smiles_list.append(Chem.MolToSmiles(mol, isomericSmiles=True))
        except Exception as e:
            print(f"Error processing SMILES {smiles}: {e}")
            continue
            
    print(f"Successfully processed SMILES: {len(remained_smiles)}")

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    output_units_num = len(targets) * per_target_output_units_num

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(
        feature_dicts['smiles_to_atom_mask'].keys())]

    # Get feature dimensions from first molecule
    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(
        [canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    # Initialize model and training components
    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, 
                       fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, 
                          weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, remained_df, targets, feature_dicts



In [ ]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    """
    Trains the model for one epoch.
    
    Args:
        model: PyTorch model to train
        dataset: Training dataset
        optimizer: Optimizer for model parameters
        loss_function: Loss function for training
        epoch: Current epoch number
        batch_size: Size of training batches
        targets: List of target column names
        feature_dicts: Feature dictionaries for molecular data
        ratio_list: Loss weighting ratios for different targets
    
    Returns:
        float: Average training loss for the epoch
    """
    model.train()
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for batch in batch_list:
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values

        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)

def eval(model, dataset, targets, feature_dicts, batch_size):
    """
    Evaluates the model on a given dataset.
    
    Args:
        model: PyTorch model to evaluate
        dataset: Dataset for evaluation
        targets: List of target column names
        feature_dicts: Feature dictionaries for molecular data
        batch_size: Size of evaluation batches
    
    Returns:
        tuple: (eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list)
    """
    model.eval()
    
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list

def train_and_evaluate(model, train_df, test_df, targets, feature_dicts, optimizer, loss_function, 
                      batch_size, num_epochs, patience=30, min_delta=0.001):
    """
    Trains and evaluates the model with early stopping.
    
    Args:
        model: PyTorch model to train
        train_df: Training dataframe
        test_df: Test dataframe
        targets: List of target column names
        feature_dicts: Feature dictionaries for molecular data
        optimizer: Optimizer for model parameters
        loss_function: Loss function for training
        batch_size: Size of training batches
        num_epochs: Maximum number of training epochs
        patience: Early stopping patience (default: 30)
        min_delta: Minimum improvement for early stopping (default: 0.001)
    
    Returns:
        dict: Training results including predictions and metrics
    """
    best_param = {
        "train_epoch": 0, 
        "val_epoch": 0, 
        "train_MSE": float('inf'), 
        "val_MSE": float('inf')
    }
    
    prediction_containers = {
        'train': {t: {'pred': [], 'actual': []} for t in targets},
        'test': {t: {'pred': [], 'actual': []} for t in targets}
    }
    
    # Create validation set from training data
    val_size = max(5, int(0.1 * len(train_df)))
    
    try:
        n_bins = min(5, max(2, len(train_df) // 3))
        train_subset, val_df = train_test_split(
            train_df, 
            test_size=val_size,
            stratify=pd.qcut(train_df[targets[0]], q=n_bins, duplicates='drop') if len(targets) > 0 else None
        )
    except ValueError:
        print("Warning: Stratification failed, using random split instead")
        train_subset, val_df = train_test_split(train_df, test_size=val_size)
    
    # Training loop with early stopping
    best_val_mse = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    
    for epoch in range(num_epochs):
        train_loss = train(model, train_subset, optimizer, loss_function, epoch, 
                          batch_size, targets, feature_dicts, ratio_list=[1.0]*len(targets))
        
        val_metrics = eval(model, val_df, targets, feature_dicts, batch_size)
        val_mae, val_mse = val_metrics[0].mean(), val_metrics[1].mean()
        
        if val_mse < best_val_mse - min_delta:
            best_val_mse = val_mse
            epochs_no_improve = 0
            best_model_state = model.state_dict()
            best_val_metrics = val_metrics
            best_param.update({
                "val_epoch": epoch,
                "val_MSE": val_mse,
                "train_epoch": epoch,
                "train_MSE": train_loss
            })
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model and evaluate
    model.load_state_dict(best_model_state)
    
    final_train_metrics = eval(model, train_subset, targets, feature_dicts, batch_size)
    final_val_metrics = best_val_metrics
    test_metrics = eval(model, test_df, targets, feature_dicts, batch_size)
    test_MAE, test_MSE, test_pred, test_actual, _ = test_metrics
    
    # Store host-level results
    host_results = {}
    for i, host in enumerate(test_df['Host'].values):
        host_results[host] = {
            target: {
                "actual": float(test_actual[target][i]),
                "predicted": float(test_pred[target][i])
            }
            for target in targets
        }
    
    # Store predictions
    for target in targets:
        prediction_containers['train'][target]['pred'].extend(final_train_metrics[2][target])
        prediction_containers['train'][target]['actual'].extend(final_train_metrics[3][target])
        
        prediction_containers['test'][target]['pred'].extend(test_metrics[2][target])
        prediction_containers['test'][target]['actual'].extend(test_metrics[3][target])
    
    # Results summary
    print("\nTraining Summary:")
    print(f"Train MSE: {np.mean(final_train_metrics[1]):.4f}")
    print(f"Val MSE: {np.mean(final_val_metrics[1]):.4f}")
    print(f"Test MSE: {np.mean(test_metrics[1]):.4f}")
    
    overall_train_mae = np.mean(final_train_metrics[0])
    overall_train_mse = np.mean(final_train_metrics[1])
    overall_test_mae = np.mean(test_metrics[0])
    overall_test_mse = np.mean(test_metrics[1])
    
    print("\nFinal Performance:")
    print(f"Train MAE: {overall_train_mae:.4f}")
    print(f"Test MAE: {overall_test_mae:.4f}")
    print(f"Train MSE: {overall_train_mse:.4f}")
    print(f"Test MSE: {overall_test_mse:.4f}")
    
    fold_result = {
        'fold': 1,
        'train_metrics': {
            'MAE': final_train_metrics[0].tolist(),
            'MSE': final_train_metrics[1].tolist()
        },
        'val_metrics': {
            'MAE': final_val_metrics[0].tolist(),
            'MSE': final_val_metrics[1].tolist()
        },
        'test_metrics': {
            'MAE': test_metrics[0].tolist(),
            'MSE': test_metrics[1].tolist()
        }
    }
    
    return {
        "host_predictions": host_results,  
        "overall_metrics": (overall_train_mae, overall_train_mse, 
                           overall_test_mae, overall_test_mse),
        "fold_results": [fold_result],
        "train_predictions": prediction_containers['train'],
        "test_predictions": prediction_containers['test']
    }


In [ ]:
# Example usage (commented out for production):
"""
raw_filename = '/notebooks/Codebase/Database/cal_abs.csv'

# Prepare the model and data
model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
    raw_filename,
    target_name='Calx', 
    targets=None, 
    batch_size=38, 
    epochs=800,
    p_dropout=0.1, 
    fingerprint_dim=150, 
    weight_decay=2, 
    learning_rate=3, 
    radius=3,
    T=2, 
    per_target_output_units_num=1
)

# Create train/test split
train_df, test_df = create_structured_absolute_ecfp_dictionary(
    remained_df, 
    split_calixarene_dict=split_calix_dict, 
    holdout_size=0.1
)

# Train and evaluate the model
results = train_and_evaluate(
    model, train_df, test_df, targets, feature_dicts, 
    optimizer, loss_function, batch_size=38, num_epochs=800, 
    patience=30, min_delta=0.001
)
"""                                      

"'raw_filename= '/notebooks/Codebase/Database/cal_abs.csv'\n\n# Prepare the model \nmodel, optimizer, loss_function, remained_df, targets, feature_dicts= prepare_model_and_data(raw_filename,\n                                                                                            target_name='Calx', targets=None, batch_size=38, epochs=800,\n                                                                                             p_dropout=0.1, fingerprint_dim=150, weight_decay=2, learning_rate=3, radius=3,\n                                                                                             T=2, per_target_output_units_num=1)\n\n\n# Prepare the test and train dataframes based on the code Jeff had\n\ntrain_df, test_df = create_structured_absolute_ecfp_dictionary(remained_df, split_calixarene_dict=split_calix_dict, holdout_size=0.1)\n\n\n#train and get the Results. \nresults = train_and_evaluate(model, train_df, test_df, targets, feature_dicts, optimizer, loss_function, \n 

In [ ]:
def run_multiple_iterations(raw_filename, split_calix_dict, holdout_size=0.1, num_iterations=20):
    """
    Runs multiple training iterations and collects results for statistical analysis.
    
    Args:
        raw_filename (str): Path to the raw data file
        split_calix_dict (dict): Dictionary with 'predictable' and 'unpredictable' host lists
        holdout_size (float): Fraction of data to use for testing (default: 0.1)
        num_iterations (int): Number of iterations to run (default: 20)
        
    Returns:
        dict: Results structured as {iteration: {host: [(pred1, actual1), (pred2, actual2), ...]}}
    """
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 
                      'H3K9me3', 'H3R2me2a', 'H3R2me2s']
    
    final_results = {}
    predictable_hosts = split_calix_dict['predictable']
    
    for i in range(num_iterations):
        print(f"Running iteration {i+1}/{num_iterations}")
        
        # Prepare model and data
        model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
            raw_filename,
            target_name='Calx', 
            targets=None, 
            batch_size=38, 
            epochs=800,
            p_dropout=0.1, 
            fingerprint_dim=150, 
            weight_decay=2, 
            learning_rate=3, 
            radius=3,
            T=2, 
            per_target_output_units_num=1
        )
        
        # Create train/test split
        train_df, test_df = create_structured_absolute_ecfp_dictionary(
            remained_df, 
            split_calixarene_dict=split_calix_dict, 
            holdout_size=holdout_size
        )
        
        # Train and evaluate
        results = train_and_evaluate(
            model, train_df, test_df, targets, feature_dicts, 
            optimizer, loss_function, batch_size=38, num_epochs=800, 
            patience=30, min_delta=0.001
        )
        
        # Extract and format results
        iteration_dict = {}
        host_predictions = results["host_predictions"]
        
        for host, predictions in host_predictions.items():
            if host in predictable_hosts:
                host_results = []
                for target in target_columns:
                    if target in predictions:
                        pred_value = predictions[target]["predicted"]
                        actual_value = predictions[target]["actual"]
                        host_results.append((pred_value, actual_value))
                iteration_dict[host] = host_results
        
        final_results[str(i)] = iteration_dict
    
    return final_results

# Configuration for multiple holdout size experiments
def run_holdout_size_experiments(raw_filename, split_calix_dict, output_dir, 
                                holdout_sizes=None, num_iterations=20):
    """
    Runs experiments with different holdout sizes and saves results.
    
    Args:
        raw_filename (str): Path to the raw data file
        split_calix_dict (dict): Dictionary with predictable/unpredictable hosts
        output_dir (str): Directory to save results
        holdout_sizes (list): List of holdout sizes to test (default: [0.05, 0.1, 0.15, 0.25, 0.5, 0.75])
        num_iterations (int): Number of iterations per holdout size (default: 20)
    """
    if holdout_sizes is None:
        holdout_sizes = [0.05, 0.1, 0.15, 0.25, 0.5, 0.75]
    
    for holdout_size in holdout_sizes:
        print(f"\nRunning experiments with holdout size: {holdout_size}")
        all_results = run_multiple_iterations(
            raw_filename, split_calix_dict, 
            holdout_size=holdout_size, num_iterations=num_iterations
        )
        
        output_filename = f"{output_dir}/20_split_{holdout_size}_AttentiveFP_abso_val.pkl"
        with open(output_filename, 'wb') as output_file:
            pickle.dump(all_results, output_file)
        print(f"Results saved to: {output_filename}")

# Example usage (commented out for production):
"""
raw_filename = 'Database/cal_abs.csv'  # Update path as needed
output_dir = 'Results'  # Update path as needed

run_holdout_size_experiments(
    raw_filename, 
    split_calix_dict, 
    output_dir, 
    holdout_sizes=[0.05, 0.1, 0.15, 0.25, 0.5, 0.75], 
    num_iterations=20
)
"""

Running iteration 1/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 32

Training Summary:
Train MSE: 2.9003
Val MSE: 3.7514
Test MSE: 0.2998

Final Performance:
Train MAE: 1.3139
Test MAE: 0.4308
Train MSE: 2.9003
Test MSE: 0.2998
Running iteration 2/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 105

Training Summary:
Train MSE: 1.4913
Val MSE: 1.5129
Test MSE: 1.1723

Final Performance:
Train MAE: 0.8580
Test MAE: 1.0106
Train MSE: 1.4913
Test MSE: 1.1723
Running iteration 3/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 33

Training Summary:
Train MSE: 2.1245
Val MSE: 6.9958
Test MSE: 1.6449

Final Performance:
Train MAE: 1.1736
Test MAE: 1.2253
Train MSE: 2.1245
Test MSE: 1.6449
Running iteration 4/20
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 64

Training Summary:
Train MSE: 2.835